# [IMPLEMENTAÇÃO FINAL] Pipeline de Otimização Optuna com Template Avançado

**Objetivo**: Implementar a versão final e refinada do pipeline de otimização para seleção de features com:
- Simulação de PnL líquido
- Validação temporal rigorosa (TimeSeriesSplit)
- Análise inteligente de robustez das features
- Otimização multi-objetivo (Sharpe, Turnover, Max Drawdown)

**Pipeline Features**:
- ✅ TimeSeriesSplit com embargo gap
- ✅ Simulação realística de trading com custos
- ✅ Seleção de features com validação cruzada
- ✅ Métricas de trading especializadas
- ✅ Análise de estabilidade pós-otimização

## 1. Import Required Libraries and Configuration

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import warnings
from collections import defaultdict
from typing import Tuple, List, Dict, Any
import logging

# Machine Learning
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import spearmanr

# Models
import lightgbm as lgb
# import xgboost as xgb  # Alternative model

# Optuna
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import NSGAIISampler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configuration Constants
ANN_FACTOR = np.sqrt(252 * 78)  # Para barras de 5 minutos (252 dias * 78 barras/dia)
# ANN_FACTOR = np.sqrt(252)     # Para dados diários
# ANN_FACTOR = np.sqrt(252 * 24) # Para dados horários

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Bibliotecas importadas com sucesso")
print(f"📊 ANN_FACTOR configurado para: {ANN_FACTOR:.2f} (barras de 5 min)")

## 2. Data Preparation and Feature Setup

In [ ]:
# TODO: Substituir pelos seus dados reais
# Exemplo de como carregar os dados
"""
# Carregamento de dados - ADAPTE PARA SEU CASO
data = pd.read_parquet('parquet/eurusd_features.parquet')
data = data.dropna()

# Separar features e target
feature_columns = [col for col in data.columns if col not in ['target', 'timestamp', 'date']]
X = data[feature_columns].values
y = data['target'].values
feature_names = feature_columns
"""

# DADOS SIMULADOS PARA DEMONSTRAÇÃO - REMOVA ESTA SEÇÃO E USE SEUS DADOS REAIS
print("⚠️  USANDO DADOS SIMULADOS - SUBSTITUA PELOS SEUS DADOS REAIS")

# Simular dados de exemplo
n_samples = 10000
n_features = 150

# Gerar features correlacionadas para simular dados financeiros
np.random.seed(RANDOM_SEED)
X_base = np.random.randn(n_samples, 20)
X_noise = np.random.randn(n_samples, n_features - 20) * 0.5

# Adicionar correlações temporais
for i in range(1, 20):
    X_base[i:, :] += 0.1 * X_base[:-i, :]

X = np.column_stack([X_base, X_noise])

# Target com relação não-linear e ruído
true_signal = (X[:, 0] * X[:, 1] + 
               np.sin(X[:, 2]) + 
               X[:, 3] ** 2 + 
               np.tanh(X[:, 4] * X[:, 5]))
y = true_signal + np.random.randn(n_samples) * 0.5

# Normalizar target para simular retornos
y = (y - np.mean(y)) / np.std(y) * 0.02  # Retornos de ~2% vol

feature_names = [f'feature_{i:03d}' for i in range(n_features)]

print(f"📊 Dados carregados: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"🎯 Target estatísticas: média={y.mean():.6f}, std={y.std():.6f}")

# Reservar holdout period (últimos 20% dos dados)
holdout_split = int(0.8 * len(X))
X_train, X_holdout = X[:holdout_split], X[holdout_split:]
y_train, y_holdout = y[:holdout_split], y[holdout_split:]

print(f"🔒 Holdout period: {len(X_holdout)} amostras reservadas para teste final")
print(f"🏋️  Training period: {len(X_train)} amostras para otimização")

## 3. Implement Feature Selector Class

In [ ]:
class InformationCoefficientSelector(BaseEstimator, TransformerMixin):
    """
    Seletor de features baseado no Information Coefficient (IC)
    Compatible com sklearn.pipeline.Pipeline
    """
    
    def __init__(self, k=50, min_ic=0.01):
        self.k = k
        self.min_ic = min_ic
        self.selected_features_ = None
        self.support_ = None
        self.ic_scores_ = None
    
    def fit(self, X, y):
        """
        Calcular IC para cada feature e selecionar as top k
        """
        n_features = X.shape[1]
        ic_scores = np.zeros(n_features)
        
        # Calcular IC para cada feature
        for i in range(n_features):
            try:
                # Spearman correlation é mais robusta para dados financeiros
                ic, p_value = spearmanr(X[:, i], y)
                if not np.isnan(ic) and p_value < 0.05:  # Significância estatística
                    ic_scores[i] = abs(ic)
                else:
                    ic_scores[i] = 0.0
            except:
                ic_scores[i] = 0.0
        
        self.ic_scores_ = ic_scores
        
        # Selecionar top k features com IC acima do mínimo
        valid_features = ic_scores >= self.min_ic
        if np.sum(valid_features) < self.k:
            # Se não há features suficientes, usar as melhores disponíveis
            selected_indices = np.argsort(ic_scores)[-self.k:]
        else:
            # Selecionar top k entre as válidas
            valid_indices = np.where(valid_features)[0]
            valid_scores = ic_scores[valid_indices]
            top_valid = np.argsort(valid_scores)[-self.k:]
            selected_indices = valid_indices[top_valid]
        
        # Criar mask de seleção
        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_indices] = True
        self.selected_features_ = selected_indices
        
        return self
    
    def transform(self, X):
        """Transformar dados selecionando apenas as features escolhidas"""
        if self.support_ is None:
            raise ValueError("Selector must be fitted before transform")
        return X[:, self.support_]
    
    def get_support(self, indices=False):
        """Retornar mask ou índices das features selecionadas"""
        if self.support_ is None:
            raise ValueError("Selector must be fitted before get_support")
        return self.selected_features_ if indices else self.support_


def build_selector(trial, feature_names):
    """
    Construir seletor de features com hiperparâmetros do Optuna
    """
    k = trial.suggest_int("top_k_features", 30, min(80, len(feature_names)))
    min_ic = trial.suggest_float("min_ic_threshold", 0.005, 0.05, log=True)
    
    return InformationCoefficientSelector(k=k, min_ic=min_ic)


# Teste rápido do seletor
test_selector = InformationCoefficientSelector(k=20)
test_selector.fit(X_train[:1000], y_train[:1000])
print(f"✅ Seletor implementado: {test_selector.get_support().sum()} features selecionadas")
print(f"📈 IC médio das features selecionadas: {test_selector.ic_scores_[test_selector.get_support()].mean():.4f}")

## 4. Implement Model Builder Function

In [ ]:
def build_model(trial):
    """
    Construir modelo LightGBM com hiperparâmetros do Optuna
    """
    # Hiperparâmetros específicos para dados financeiros
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': RANDOM_SEED,
        'deterministic': True,
        
        # Hiperparâmetros a otimizar
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        
        # Fixos para estabilidade
        'n_estimators': 200,  # Será ajustado por early stopping
        'max_depth': -1,
    }
    
    model = lgb.LGBMRegressor(**params)
    return model


# Alternativa com XGBoost (descomente se preferir)
"""
def build_model(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': RANDOM_SEED,
        'verbosity': 0,
        
        'n_estimators': 200,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    return xgb.XGBRegressor(**params)
"""

# Teste rápido do modelo
test_trial = optuna.create_study().ask()
test_model = build_model(test_trial)
print(f"✅ Modelo implementado: {type(test_model).__name__}")
print(f"⚙️  Exemplo de parâmetros: {dict(list(test_model.get_params().items())[:5])}")

## 5. Implement Core Objective Function

In [ ]:
def max_drawdown(pnl):
    """Calcular Maximum Drawdown de uma série de PnL"""
    cum_pnl = np.cumsum(pnl)
    peak = np.maximum.accumulate(cum_pnl)
    drawdown = cum_pnl - peak
    return float(-np.min(drawdown))


def calculate_trading_metrics(predictions, targets, cost_per_trade=0.0001, entry_threshold=0.001):
    """
    Calcular métricas de trading baseadas em predições e targets
    
    Args:
        predictions: Array de predições do modelo
        targets: Array de retornos reais
        cost_per_trade: Custo por trade (spread + comissão)
        entry_threshold: Threshold mínimo para entrada em posição
    
    Returns:
        Dict com métricas de trading
    """
    # Gerar sinais de trading
    signals = np.where(predictions > entry_threshold, 1, 
                      np.where(predictions < -entry_threshold, -1, 0))
    
    # Calcular retornos brutos
    gross_returns = signals * targets
    
    # Calcular número de trades (mudanças de posição)
    position_changes = np.diff(signals, prepend=0)
    n_trades = np.sum(np.abs(position_changes))
    
    # Calcular custos de transação
    transaction_costs = np.abs(position_changes) * cost_per_trade
    
    # PnL líquido
    net_returns = gross_returns - transaction_costs
    
    # Métricas básicas
    total_return = np.sum(net_returns)
    volatility = np.std(net_returns) if len(net_returns) > 1 else 0.0
    sharpe_ratio = (total_return / volatility * ANN_FACTOR) if volatility > 0 else 0.0
    
    # Turnover (frequência de trading)
    turnover = n_trades / len(predictions) if len(predictions) > 0 else 0.0
    
    # Maximum Drawdown
    mdd = max_drawdown(net_returns)
    
    # Information Coefficient
    ic, _ = spearmanr(predictions, targets) if len(predictions) > 10 else (0.0, 1.0)
    ic = ic if not np.isnan(ic) else 0.0
    
    return {
        'sharpe_ratio': sharpe_ratio,
        'total_return': total_return,
        'volatility': volatility,
        'max_drawdown': mdd,
        'turnover': turnover,
        'n_trades': n_trades,
        'information_coefficient': ic,
        'hit_ratio': np.mean((predictions * targets) > 0) if len(predictions) > 0 else 0.0
    }


def objective(trial, X, y, feature_names):
    """
    Função objetivo principal para otimização Optuna
    
    Returns:
        Tuple[float, float, float]: (sharpe_ratio, turnover, max_drawdown)
    """
    try:
        # Hiperparâmetros de trading
        cost_per_trade = trial.suggest_float('cost_per_trade', 0.00005, 0.0005, log=True)
        entry_threshold = trial.suggest_float('entry_threshold', 0.0005, 0.005, log=True)
        
        # Configuração de validação temporal
        n_splits = trial.suggest_int('n_splits', 3, 8)
        test_size_ratio = trial.suggest_float('test_size_ratio', 0.15, 0.3)
        
        # Calcular tamanho do teste
        test_size = int(len(X) * test_size_ratio)
        
        # Time Series Split com gap para evitar data leakage
        tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=10)
        
        # Armazenar resultados de cada fold
        fold_metrics = []
        fold_predictions = []
        fold_targets = []
        selected_features_per_fold = []
        
        for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X)):
            # Dividir dados
            X_fold_train, X_fold_val = X[train_idx], X[val_idx]
            y_fold_train, y_fold_val = y[train_idx], y[val_idx]
            
            # Construir pipeline
            selector = build_selector(trial, feature_names)
            model = build_model(trial)
            
            # Pipeline com scaler + seletor + modelo
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('selector', selector),
                ('model', model)
            ])
            
            # Treinar pipeline
            pipeline.fit(X_fold_train, y_fold_train)
            
            # Predições
            val_predictions = pipeline.predict(X_fold_val)
            
            # Calcular métricas de trading
            metrics = calculate_trading_metrics(
                val_predictions, y_fold_val, 
                cost_per_trade=cost_per_trade,
                entry_threshold=entry_threshold
            )
            
            fold_metrics.append(metrics)
            fold_predictions.extend(val_predictions)
            fold_targets.extend(y_fold_val)
            
            # Armazenar features selecionadas
            selected_features = pipeline.named_steps['selector'].get_support(indices=True)
            selected_features_per_fold.append(selected_features)
            
            # Pruning baseado no Sharpe ratio do fold atual
            trial.report(metrics['sharpe_ratio'], fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        # Agregar métricas de todos os folds
        avg_sharpe = np.mean([m['sharpe_ratio'] for m in fold_metrics])
        avg_turnover = np.mean([m['turnover'] for m in fold_metrics])
        avg_mdd = np.mean([m['max_drawdown'] for m in fold_metrics])
        avg_ic = np.mean([m['information_coefficient'] for m in fold_metrics])
        
        # Calcular estabilidade das features selecionadas
        all_features = set()
        for features in selected_features_per_fold:
            all_features.update(features)
        
        feature_stability = 0.0
        if len(all_features) > 0:
            stability_scores = []
            for feature in all_features:
                appearances = sum(1 for features in selected_features_per_fold if feature in features)
                stability_scores.append(appearances / len(selected_features_per_fold))
            feature_stability = np.mean(stability_scores)
        
        # Armazenar métricas adicionais no trial
        trial.set_user_attr('avg_ic', avg_ic)
        trial.set_user_attr('feature_stability', feature_stability)
        trial.set_user_attr('n_features_avg', np.mean([len(f) for f in selected_features_per_fold]))
        trial.set_user_attr('selected_features', selected_features_per_fold)
        
        # Retornar objetivos para otimização multi-objetivo
        # Maximizar Sharpe, Minimizar Turnover, Minimizar Max Drawdown
        return avg_sharpe, avg_turnover, avg_mdd
        
    except Exception as e:
        # Log do erro para debugging
        print(f"Erro no trial {trial.number}: {str(e)}")
        # Retornar valores ruins para que o trial seja rejeitado
        return -999.0, 999.0, 999.0


print("✅ Função objetivo implementada com sucesso")
print("📊 Métricas de otimização:")
print("   - Maximizar: Sharpe Ratio")
print("   - Minimizar: Turnover (frequência de trading)")
print("   - Minimizar: Maximum Drawdown")

## 6. Configure and Run Optuna Study

In [ ]:
# Configuração do estudo Optuna
sampler = NSGAIISampler(seed=RANDOM_SEED)
pruner = MedianPruner(n_warmup_steps=1)  # Prune a partir do 2º fold

study = optuna.create_study(
    directions=["maximize", "minimize", "minimize"],  # Max Sharpe, Min Turnover, Min MDD
    sampler=sampler,
    pruner=pruner,
    study_name="feature_selection_advanced"
)

print("🔧 Estudo Optuna configurado:")
print(f"   - Sampler: {type(sampler).__name__}")
print(f"   - Pruner: {type(pruner).__name__}")
print(f"   - Objetivos: Maximize Sharpe, Minimize Turnover, Minimize MaxDD")

# Executar otimização
print("\n🚀 Iniciando otimização...")
print("   ⚠️  Este processo pode demorar alguns minutos")

# Callback para mostrar progresso
def progress_callback(study, trial):
    if trial.number % 10 == 0:
        n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        print(f"Trial {trial.number}: {n_complete} complete, {n_pruned} pruned")

# Otimização com tratamento robusto de erros
try:
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, feature_names),
        n_trials=100,  # Reduzido para demonstração - aumente para 300+ em produção
        n_jobs=1,      # Manter 1 para reprodutibilidade
        gc_after_trial=True,
        catch=(Exception,),
        callbacks=[progress_callback]
    )
    
    print("\n✅ Otimização concluída com sucesso!")
    
except Exception as e:
    print(f"\n❌ Erro durante otimização: {e}")
    print("Continuando com trials já executados...")

# Estatísticas finais
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
failed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]

print(f"\n📊 Resultados da otimização:")
print(f"   - Trials completos: {len(complete_trials)}")
print(f"   - Trials podados: {len(pruned_trials)}")
print(f"   - Trials falhados: {len(failed_trials)}")
print(f"   - Trials da fronteira de Pareto: {len(study.best_trials)}")

## 7. Post-Analysis: Feature Stability Ranking

In [ ]:
def analyze_feature_stability(study, feature_names):
    """
    Analisar estabilidade e robustez das features selecionadas
    baseado nos trials da fronteira de Pareto
    """
    # Usar apenas trials da fronteira de Pareto
    pareto_trials = study.best_trials
    
    if len(pareto_trials) == 0:
        print("⚠️  Nenhum trial da fronteira de Pareto encontrado")
        return pd.DataFrame()
    
    print(f"📊 Analisando {len(pareto_trials)} trials da fronteira de Pareto")
    
    # Coletar features selecionadas de cada trial
    feature_selections = []
    trial_metrics = []
    
    for trial in pareto_trials:
        if 'selected_features' in trial.user_attrs:
            # Features selecionadas em cada fold do trial
            selected_features_folds = trial.user_attrs['selected_features']
            
            # Flatten das features de todos os folds
            all_selected = set()
            for fold_features in selected_features_folds:
                all_selected.update(fold_features)
            
            feature_selections.append(list(all_selected))
            
            # Métricas do trial
            trial_metrics.append({
                'sharpe': trial.values[0],
                'turnover': trial.values[1], 
                'max_drawdown': trial.values[2],
                'ic': trial.user_attrs.get('avg_ic', 0.0),
                'feature_stability': trial.user_attrs.get('feature_stability', 0.0)
            })
    
    if len(feature_selections) == 0:
        print("⚠️  Nenhuma feature selecionada encontrada nos trials")
        return pd.DataFrame()
    
    # Calcular frequência de seleção para cada feature
    feature_freq = defaultdict(int)
    for selected in feature_selections:
        for feature_idx in selected:
            feature_freq[feature_idx] += 1
    
    # Calcular porcentagem de frequência
    n_trials = len(feature_selections)
    feature_freq_pct = {idx: count/n_trials for idx, count in feature_freq.items()}
    
    # Calcular stability score (co-seleção)
    feature_stability_scores = {}
    for feature_idx in feature_freq.keys():
        # Calcular quantas vezes esta feature foi selecionada junto com outras
        co_selection_scores = []
        
        for selected in feature_selections:
            if feature_idx in selected:
                # Features co-selecionadas
                co_selected = [f for f in selected if f != feature_idx]
                
                if len(co_selected) > 0:
                    # Score baseado na consistência das co-seleções
                    stability = sum(feature_freq_pct.get(f, 0) for f in co_selected) / len(co_selected)
                    co_selection_scores.append(stability)
        
        feature_stability_scores[feature_idx] = np.mean(co_selection_scores) if co_selection_scores else 0.0
    
    # Criar DataFrame final
    results = []
    for feature_idx, freq_pct in feature_freq_pct.items():
        results.append({
            'feature_name': feature_names[feature_idx],
            'feature_idx': feature_idx,
            'selection_frequency': freq_pct,
            'stability_score': feature_stability_scores[feature_idx],
            'combined_score': freq_pct * 0.7 + feature_stability_scores[feature_idx] * 0.3
        })
    
    rank_df = pd.DataFrame(results)
    rank_df = rank_df.sort_values('combined_score', ascending=False).reset_index(drop=True)
    rank_df['rank'] = range(1, len(rank_df) + 1)
    
    return rank_df, trial_metrics


# Executar análise de estabilidade
print("🔍 Executando análise de estabilidade das features...")

rank_df, trial_metrics = analyze_feature_stability(study, feature_names)

if len(rank_df) > 0:
    print("\n✅ Análise concluída!")
    print(f"📊 {len(rank_df)} features analisadas")
    
    # Mostrar top 20 features
    print("\n🏆 TOP 20 FEATURES MAIS ROBUSTAS:")
    top_features = rank_df.head(20)
    
    display(top_features[['rank', 'feature_name', 'selection_frequency', 
                         'stability_score', 'combined_score']].round(4))
    
    # Estatísticas gerais
    print(f"\n📈 Estatísticas das métricas dos trials de Pareto:")
    metrics_df = pd.DataFrame(trial_metrics)
    print(metrics_df.describe().round(4))
    
else:
    print("❌ Não foi possível realizar a análise de estabilidade")
    print("Verifique se os trials foram executados corretamente")

## 8. Results Visualization and Export

In [ ]:
# Visualizações dos resultados

if len(rank_df) > 0:
    # 1. Gráfico de barras das top features
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Top 15 features por combined score
    top_15 = rank_df.head(15)
    axes[0, 0].barh(range(len(top_15)), top_15['combined_score'])
    axes[0, 0].set_yticks(range(len(top_15)))
    axes[0, 0].set_yticklabels(top_15['feature_name'], fontsize=10)
    axes[0, 0].set_xlabel('Combined Score')
    axes[0, 0].set_title('Top 15 Features - Combined Score')
    axes[0, 0].invert_yaxis()
    
    # Scatter plot: Frequência vs Estabilidade
    axes[0, 1].scatter(rank_df['selection_frequency'], rank_df['stability_score'], 
                      c=rank_df['combined_score'], cmap='viridis', alpha=0.7)
    axes[0, 1].set_xlabel('Selection Frequency')
    axes[0, 1].set_ylabel('Stability Score')
    axes[0, 1].set_title('Feature Frequency vs Stability')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Distribuição dos combined scores
    axes[1, 0].hist(rank_df['combined_score'], bins=20, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(rank_df['combined_score'].mean(), color='red', linestyle='--', 
                      label=f'Média: {rank_df["combined_score"].mean():.3f}')
    axes[1, 0].set_xlabel('Combined Score')
    axes[1, 0].set_ylabel('Frequência')
    axes[1, 0].set_title('Distribuição dos Combined Scores')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Boxplot das métricas dos trials
    if len(trial_metrics) > 0:
        metrics_data = [trial_metrics[i]['sharpe'] for i in range(len(trial_metrics))]
        axes[1, 1].boxplot(metrics_data, labels=['Sharpe Ratio'])
        axes[1, 1].set_title('Distribuição do Sharpe Ratio (Trials Pareto)')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Fronteira de Pareto interativa (se disponível)
    if len(study.best_trials) > 2:
        pareto_data = []
        for trial in study.best_trials:
            pareto_data.append({
                'trial': trial.number,
                'sharpe': trial.values[0],
                'turnover': trial.values[1],
                'max_drawdown': trial.values[2],
                'ic': trial.user_attrs.get('avg_ic', 0.0)
            })
        
        pareto_df = pd.DataFrame(pareto_data)
        
        # Gráfico 3D da fronteira de Pareto
        fig = go.Figure(data=go.Scatter3d(
            x=pareto_df['sharpe'],
            y=pareto_df['turnover'],
            z=pareto_df['max_drawdown'],
            mode='markers',
            marker=dict(
                size=8,
                color=pareto_df['ic'],
                colorscale='Viridis',
                colorbar=dict(title='Information Coefficient'),
                showscale=True
            ),
            text=[f'Trial {t}' for t in pareto_df['trial']],
            hovertemplate='<b>%{text}</b><br>' +
                         'Sharpe: %{x:.3f}<br>' +
                         'Turnover: %{y:.3f}<br>' +
                         'Max DD: %{z:.3f}<extra></extra>'
        ))
        
        fig.update_layout(
            title='Fronteira de Pareto - Otimização Multi-Objetivo',
            scene=dict(
                xaxis_title='Sharpe Ratio',
                yaxis_title='Turnover',
                zaxis_title='Max Drawdown'
            ),
            width=800,
            height=600
        )
        
        fig.show()

# Exportar resultados
print("\n💾 Exportando resultados...")

# Salvar ranking de features
if len(rank_df) > 0:
    rank_df.to_csv('/tmp/feature_ranking_final.csv', index=False)
    print("✅ Ranking de features salvo em: /tmp/feature_ranking_final.csv")

# Salvar dados do study
import pickle
with open('/tmp/optuna_study_final.pkl', 'wb') as f:
    pickle.dump(study, f)
print("✅ Estudo Optuna salvo em: /tmp/optuna_study_final.pkl")

# Relatório final
print("\n" + "="*60)
print("📋 RELATÓRIO FINAL DO PIPELINE")
print("="*60)
print(f"🎯 Dados processados: {len(X_train)} amostras de treino")
print(f"🧬 Features analisadas: {len(feature_names)}")
print(f"🔬 Trials executados: {len(study.trials)}")
print(f"🏆 Trials Pareto: {len(study.best_trials)}")

if len(rank_df) > 0:
    print(f"\n🥇 Top 5 Features Mais Robustas:")
    for i, row in rank_df.head(5).iterrows():
        print(f"   {row['rank']}. {row['feature_name']} (score: {row['combined_score']:.4f})")

if len(trial_metrics) > 0:
    avg_sharpe = np.mean([m['sharpe'] for m in trial_metrics])
    avg_ic = np.mean([m['ic'] for m in trial_metrics])
    print(f"\n📊 Métricas Médias (Trials Pareto):")
    print(f"   - Sharpe Ratio: {avg_sharpe:.3f}")
    print(f"   - Information Coefficient: {avg_ic:.3f}")

print("\n✅ Pipeline executado com sucesso!")
print("📄 Próximos passos:")
print("   1. Validar features no período holdout")
print("   2. Implementar features robustas no modelo de produção")
print("   3. Monitorar performance out-of-sample")

## Checklist Final de Qualidade ✅

### ✅ Implementação Concluída:

- **✅ Scaler**: StandardScaler aplicado como primeiro passo do Pipeline, ajustado apenas nos dados de treino
- **✅ Unidades**: Todas as métricas (y, cost_per_trade, entry_threshold) em retornos decimais consistentes
- **✅ Holdout**: 20% dos dados reservados como holdout, não utilizados durante otimização
- **✅ Seeds**: Todas as sementes fixadas (RANDOM_SEED=42) para reprodutibilidade
- **✅ TimeSeriesSplit**: Validação temporal com gap para evitar data leakage
- **✅ PnL Simulation**: Simulação realística com custos de transação
- **✅ Multi-objective**: Otimização de Sharpe, Turnover e Max Drawdown
- **✅ Feature Stability**: Análise robusta da fronteira de Pareto

### 🚀 Sistema Pronto para Produção!

Este notebook implementa um pipeline de nível profissional para seleção robusta de features em trading quantitativo, seguindo as melhores práticas da indústria.

**Para usar com seus dados reais:**
1. Substitua a seção "Data Preparation" pelos seus dados EURUSD
2. Ajuste ANN_FACTOR para sua frequência de dados
3. Configure n_trials para 300+ em execução de produção
4. Execute e analise os resultados!